# Pattern 4: Multi-Agent Supervisor Routing

This notebook demonstrates a supervisor that routes work between specialist agents.

In [ ]:
from typing import TypedDict, Literal

from langchain_ollama import ChatOllama
from langgraph.graph import END, START, StateGraph

MODEL_NAME = "qwen3.5:2b"
llm = ChatOllama(model=MODEL_NAME, temperature=0)

In [ ]:
class MultiAgentState(TypedDict):
    question: str
    research_notes: str
    draft_answer: str
    final_answer: str
    route: Literal['researcher', 'writer', 'finish']

def supervisor_node(state: MultiAgentState) -> MultiAgentState:
    route_prompt = f"""
You are a supervisor. Choose exactly one next route: researcher, writer, or finish.
Question: {state['question']}
Current research notes length: {len(state['research_notes'])}
Current draft length: {len(state['draft_answer'])}

Rules:
- If research notes are weak/empty, choose researcher.
- If research exists but no draft, choose writer.
- If draft exists and is enough, choose finish.
Return only one word.
"""
    choice = str(llm.invoke(route_prompt).content).strip().lower()
    if 'researcher' in choice:
        state['route'] = 'researcher'
    elif 'writer' in choice:
        state['route'] = 'writer'
    else:
        state['route'] = 'finish'
    return state

def researcher_node(state: MultiAgentState) -> MultiAgentState:
    prompt = f"""
Collect concise research notes (5 bullets max) for this question:
{state['question']}
"""
    state['research_notes'] = str(llm.invoke(prompt).content)
    return state

def writer_node(state: MultiAgentState) -> MultiAgentState:
    prompt = f"""
Use these notes to draft a clear answer.
Question: {state['question']}
Notes:
{state['research_notes']}
"""
    state['draft_answer'] = str(llm.invoke(prompt).content)
    return state

def finalize_node(state: MultiAgentState) -> MultiAgentState:
    if state['draft_answer']:
        state['final_answer'] = state['draft_answer']
    else:
        state['final_answer'] = state['research_notes']
    return state

def route_from_supervisor(state: MultiAgentState) -> str:
    return state['route']

In [ ]:
builder = StateGraph(MultiAgentState)
builder.add_node('supervisor', supervisor_node)
builder.add_node('researcher', researcher_node)
builder.add_node('writer', writer_node)
builder.add_node('finalize', finalize_node)

builder.add_edge(START, 'supervisor')
builder.add_conditional_edges('supervisor', route_from_supervisor, {
    'researcher': 'researcher',
    'writer': 'writer',
    'finish': 'finalize',
})
builder.add_edge('researcher', 'supervisor')
builder.add_edge('writer', 'supervisor')
builder.add_edge('finalize', END)

graph = builder.compile()

initial = {
    'question': 'What are practical first projects to learn agentic AI on a laptop?',
    'research_notes': '',
    'draft_answer': '',
    'final_answer': '',
    'route': 'researcher',
}

result = graph.invoke(initial)
result['final_answer']

## Notes
This pattern scales to many specialists (tools, retrieval, coding, validation) with a stronger supervisor policy.